# Notebook 28 — contact sheets for the heuristic MCQ-like items

**Everything on these sheets is labelled MCQ-*like*, never MCQ.** The set comes
from `strict_v2._looks_mcq`, a loose detector built for a review queue, and the
whole point of this notebook is to let a human decide which of its hits are
real.

**Why it needs auditing.** The detector fires on two things: the word *option*
anywhere in the question, truth span or answer field, or an `(a)…(b)` pattern
within 60 characters. The second trigger is unsound on this corpus, and the
matched text says so plainly:

| item | what the regex matched | what it actually is |
|---|---|---|
| 82, 234 | `P(A) = P(B)`, `P(A) + P(B)` | probability notation |
| 117 | `P(A) = \frac{7}{13} …P(B)` | probability notation |
| 195 | `f(a) = f(b)` | function notation |
| 170, 237 | `(d)$. Similarly, $(ii)$ matches $(c)` | a **matching** exercise |

**The audit is deliberately TWO-SIDED, and this is the part that matters.**
Reviewing only the flagged items can find false positives and nothing else, so
a corrected count would be biased downward by construction. That is the exact
one-sided mistake made earlier in this project: the 104-item `genuinely_wrong`
census could only surface false negatives, and a whole separate correct-side
spot check had to be built afterwards. So this notebook renders **two** folders:

| sheet | n | what to do |
|---|---|---|
| `mcq_flagged_p*.png` | 58 | **CONFIRM or REJECT** — is this really multiple choice? |
| `mcq_candidate_missed_p*.png` | 19 | **RECOVER** — the detector said no, but the question carries a choice list |

Items are ordered strongest-evidence-first inside each group, so the doubtful
ones cluster at the end of the flagged sheets rather than being scattered.

**What is at stake.** MCQ accuracy is currently reported as 82.8%, and it moves
to **88.5%** if the six weak-trigger items are dropped or **78.9%** if they are
dropped and all nineteen candidates are added. The *direction* (MCQ far above
free response, 38.4%) is stable; the magnitude is not.

No GPU. Reads the dataset and one results CSV; runs no generation and changes
no scorer rule.

In [ ]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and an existing results CSV, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. This cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.dataset_profile
import pilot.plotting
import pilot.rescore
import pilot.strict_v2

print(f"pilot imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert hasattr(pilot.dataset_profile, "mcq_review_set"), (
    "the repo clone predates notebook 28's library code -- push "
    "pilot/dataset_profile.py and re-run this cell. A green local dry run "
    "does NOT cover this: the dry run uses the working tree, Colab uses the "
    "remote.")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain "
    "text, which changes the spans and labels burned into these captions.")
print("SymPy LaTeX parser OK")

In [ ]:
import pandas as pd

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5
PER_PAGE = 9                      # must match mcq_review_set's per_page

run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
assert len(run) == 300, f"expected 300 rows, got {len(run)}"
print(f"run: {len(run)} rows, model={run['model_id'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Attaching a caption to the wrong page
# would make this audit worse than useless.
assert len(sample) == len(run), f"{len(sample)} items vs {len(run)} rows"
bad = [i for i in range(len(run))
       if sample[i]["orig_q"].strip() != str(run.iloc[i]["orig_q"]).strip()]
assert not bad, (
    f"{len(bad)} rows where the rebuilt sample's question does not match the "
    f"CSV's (first: {bad[:5]}). Images would be attached to the wrong rows.")
print("sample order matches the CSV on all 300 rows -- images are index-aligned")

v1 = pilot.rescore.rescore_run(run, "strict_v1", progress=True)["transcription_correct"].astype(bool)
v2s = pilot.strict_v2.rescore_v2(run, progress=True)
review = pilot.dataset_profile.mcq_review_set(run, v1, v2s, per_page=PER_PAGE)

# The committed manifest and these sheets must describe the same set. If the
# repo copy disagrees, the CSV a reviewer codes against is not what they see.
committed = pd.read_csv("repo/reference/audit/mcq_like_review_20260812.csv")
assert sorted(committed["item_id"]) == sorted(review["item_id"]), (
    "the rebuilt review set differs from the committed manifest -- do not "
    "code against sheets whose CSV does not match them")
print(f"\nreview set: {len(review)} items, matches the committed manifest")
print(review.groupby(["mcq_group", "presort"]).size().to_string())

In [ ]:
# Contact sheets, same format as notebook 17 section 6 and notebook 23.
# CAPTIONED PNGs are what makes a folder reviewable IN DRIVE -- bare per-item
# images are a wall of uncaptioned thumbnails, and HTML does not render in
# Drive's preview.
import matplotlib.pyplot as plt

TITLES = {
    "flagged_mcq_like":
        "HEURISTIC MCQ-LIKE - is each really multiple choice? (strongest evidence first)",
    "candidate_missed":
        "NOT flagged by the detector, but the question carries a choice list - RECOVER?",
}
STEM = {"flagged_mcq_like": "mcq_flagged",
        "candidate_missed": "mcq_candidate_missed"}

SHEET_DIR = f"{PROJECT_DIR}/figures/mcq_like_review"
os.makedirs(SHEET_DIR, exist_ok=True)

written = {}
for group, sub in review.groupby("mcq_group", sort=False):
    items = sub["item_id"].astype(int).tolist()
    figs = pilot.plotting.contact_sheet(
        [sample[i]["image"] for i in items],
        [pilot.dataset_profile.mcq_caption(r) for _, r in sub.iterrows()],
        ncols=3, per_page=PER_PAGE, cell_height=4.8, caption_fontsize=6.0,
        title=TITLES[group])
    paths = []
    for page, fig in enumerate(figs, 1):
        path = f"{SHEET_DIR}/{STEM[group]}_p{page}.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor())
        plt.close(fig)
        paths.append(path)
    written[group] = paths
    print(f"{group}: {len(items)} items -> {len(figs)} page(s)")

# The manifest names a sheet file per item; every one of those files must
# exist, or a reviewer follows the CSV to a page that was never rendered.
missing = sorted({f for f in review["contact_sheet_file"]
                  if not os.path.exists(f"{SHEET_DIR}/{f}")})
assert not missing, f"manifest points at sheets that were not written: {missing}"
print(f"\nall {review['contact_sheet_file'].nunique()} sheet files referenced "
      "by the manifest exist")

# Write the COMMITTED manifest, not the freshly rebuilt one. The rebuild
# ships its human columns empty by design, so copying it to Drive and back
# would silently erase the completed audit. `committed` is the same 77 rows in
# the same order (asserted above) plus the codings.
committed.to_csv(f"{SHEET_DIR}/mcq_like_review_20260812.csv", index=False)
n_coded = int((committed["confirmed_mcq"].astype(str) != "").sum())
print(f"manifest -> {SHEET_DIR}/mcq_like_review_20260812.csv "
      f"({n_coded}/{len(committed)} already coded)")

In [ ]:
# What the sheets are for, and what the numbers do if the count changes.
sens = pilot.dataset_profile.mcq_accuracy_sensitivity(
    pilot.dataset_profile.add_derived_groupings(
        pilot.dataset_profile.item_profile(run, v1, v2s)),
    review)
print(f"heuristic MCQ-like        : {sens['n_flagged']}")
print(f"  of which weak trigger   : {sens['n_weak_trigger']}  "
      "(no 'option' word anywhere; flagged only by the (a)...(b) regex)")
print(f"candidate MISSED by it    : {sens['n_candidate_missed']}")
print()
for key, label in (("as_reported", "as reported"),
                   ("drop_weak_trigger", "drop weak-trigger items"),
                   ("drop_weak_add_missed", "drop weak + add all candidates")):
    b = sens[key]
    print(f"  {label:32s} n={b['n']:3d}  strict_v1 accuracy {b['accuracy']:.1%}")
print(f"\nrange {sens['range'][0]:.1%} - {sens['range'][1]:.1%}; quotable={sens['quotable']}")
print(f"why: {sens['why']}")

print("\n" + "=" * 70)
print("open: My Drive > uncertainty-math-vlm > figures > mcq_like_review")
print("""
Fill in `confirmed_mcq` per row of the manifest CSV:
  yes        - the page really is a multiple-choice question
  no         - it is not (a matching exercise, probability notation, a
               multi-part question numbered (a)/(b), ...)
  unclear    - cannot tell from the page

`reviewer_note` is free text. Your reading of the IMAGE overrides every
automatic field burned into the caption.

Both groups need coding. The `candidate_missed` sheets are the half that can
move the count UP -- skipping them makes the corrected MCQ set biased
downward by construction.
""")